# QLoRA Fine-tuning: IDF Query Model

Fine-tunes `Qwen2.5-Coder-7B-Instruct` on IDF proto generation using Unsloth.

**Steps:**
1. Install dependencies
2. Load model in 4-bit
3. Apply LoRA adapters
4. Load training data
5. Train (3 epochs, ~30-60 min on T4)
6. Export to GGUF for Ollama

**Prerequisites:** Upload `training_data.jsonl` to this Colab runtime before running.

In [ ]:
# Step 0: Install dependencies
!pip install -q unsloth transformers datasets trl peft accelerate bitsandbytes

In [ ]:
# Step 1: Load base model with 4-bit quantization
from unsloth import FastLanguageModel
import torch, gc, os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_NAME = "unsloth/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LENGTH = 1024  # Reduced from 2048 — our protos are short

print("Loading base model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
print(f"Model loaded: {MODEL_NAME}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1024**3:.1f} GB used")

In [ ]:
# Step 2: Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("LoRA applied: r=16, alpha=32")
model.print_trainable_parameters()

In [ ]:
# Step 3: Load and format training data
from datasets import load_dataset

TRAINING_FILE = "training_data.jsonl"

dataset = load_dataset("json", data_files=TRAINING_FILE, split="train")
print(f"Dataset loaded: {len(dataset)} examples")

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat, num_proc=2)

# Filter out examples that are too long (prevents OOM on outliers)
def is_short_enough(example):
    return len(tokenizer.encode(example["text"])) <= MAX_SEQ_LENGTH

dataset = dataset.filter(is_short_enough, num_proc=2)
print(f"After filtering to max {MAX_SEQ_LENGTH} tokens: {len(dataset)} examples")
print(f"Sample (first 400 chars):\n{dataset[0]['text'][:400]}")

In [ ]:
# Step 4: Train (memory-optimized for T4 15GB)
import gc
import torch
from trl import SFTTrainer
from transformers import TrainingArguments

gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=1,       # Reduced from 4 to fit T4
        gradient_accumulation_steps=16,      # Effective batch = 16 (same as before)
        warmup_steps=10,                     # Fixed deprecation warning
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="./idf_query_output",
        save_strategy="epoch",
        report_to="none",
        gradient_checkpointing=True,         # Trades compute for memory
        gradient_checkpointing_kwargs={"use_reentrant": False},
    ),
)

print(f"GPU memory before training: {torch.cuda.memory_allocated()/1024**3:.1f} GB")
print("Starting training...")
stats = trainer.train()
print(f"\nDone! Loss: {stats.training_loss:.4f}, Steps: {stats.global_step}, Time: {stats.metrics['train_runtime']:.0f}s")

In [ ]:
# Step 5: Save LoRA adapter
model.save_pretrained("./idf_query_lora")
tokenizer.save_pretrained("./idf_query_lora")
print("LoRA adapter saved to: ./idf_query_lora")

In [ ]:
# Step 6: Export to GGUF for Ollama
print("Exporting to GGUF (q4_k_m)...")
model.save_pretrained_gguf(
    "./idf_query_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF exported to: ./idf_query_gguf/")

In [ ]:
# Step 7: Generate Ollama Modelfile
import os

gguf_files = [f for f in os.listdir("./idf_query_gguf") if f.endswith(".gguf")]
gguf_filename = gguf_files[0] if gguf_files else "unsloth.Q4_K_M.gguf"

modelfile = f"""FROM ./{gguf_filename}

TEMPLATE \"\"\"{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{- end }}}}
<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
\"\"\"

PARAMETER stop \"<|im_end|>\"
PARAMETER stop \"<|im_start|>\"
PARAMETER temperature 0
PARAMETER num_predict 512
"""

with open("./idf_query_gguf/Modelfile", "w") as f:
    f.write(modelfile)

print("Modelfile written to: ./idf_query_gguf/Modelfile")
print()
print("=" * 50)
print("NEXT STEPS:")
print("=" * 50)
print("1. Download the ./idf_query_gguf/ folder")
print("2. On your machine, run:")
print("   cd idf_query_gguf")
print("   ollama create idf-query-7b -f Modelfile")
print("3. Update config.py:")
print('   CHAT_MODEL = \"idf-query-7b\"')

In [ ]:
# Step 8: Quick inference test
FastLanguageModel.for_inference(model)

test_queries = [
    "get all VMs",
    "create a vm named test_001",
    "update vm test_001 setting power_state to on",
    "fetch vm details for test_001",
    "delete vm entity test_001",
]

for query in test_queries:
    messages = [
        {"role": "system", "content": "You are an IDF query generator. Output API method and proto."},
        {"role": "user", "content": query},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        input_ids=inputs, max_new_tokens=256, temperature=0.0, do_sample=False,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nQ: {query}")
    print(f"A: {response[:250]}")
    print("-" * 40)

In [ ]:
# Step 9: Download the GGUF (Colab)
# Zip for easy download
!zip -r idf_query_gguf.zip ./idf_query_gguf/
print("\nDownload idf_query_gguf.zip from the Files panel on the left.")